Quantum Lattice Boltzmann (QLBM) : principe, utilité, démo en Qiskit
1. À quoi ça sert ?

La méthode de Boltzmann sur réseau (Lattice Boltzmann Method, LBM) est une façon de simuler des fluides (air autour d’une aile, écoulement dans un moteur, transport de chaleur ou de polluants). Au lieu de résoudre directement Navier–Stokes, on suit des populations de particules fictives $f_i(x,t)$ qui se déplacent sur une grille selon quelques vitesses discrètes $c_i$ et se “mélangent” localement.

Pourquoi le quantique ? Un état de $n$ qubits a $2^n$ amplitudes. Une grille de $N=2^n$ points peut donc être stockée dans seulement $\log_2 N$ qubits. Si on sait faire avancer cet état en temps avec des circuits peu profonds, on espère une compression exponentielle de la mémoire et un gain en temps pour la CFD (aérodynamique, combustion, météo). Toulouse étant un pôle aéronautique et spatial, c’est un cas d’usage naturel.

La limite à garder en tête (très probablement abordée en hackathon) :

Atout	Difficulté
Le transport (streaming) est unitaire : bon pour un circuit quantique	La collision est non unitaire (dissipation) et, pour un vrai fluide, non linéaire
Mémoire en $\log N$	Préparer l’état initial et lire le résultat coûtent cher (lire les $N$ valeurs détruit le gain)
	Les méthodes à post-sélection ont une probabilité de succès qui décroît avec le temps

Les pistes de recherche que je connais : linéarisation de Carleman pour le terme non linéaire, combinaison linéaire d’unitaires (LCU), schémas hybrides quantique/classique avec réencodage. Je ne peux pas vérifier ici les références précises, donc vérifie les articles (par exemple Budinski ; Schalkers et Möller ; Itani, Sreenivasan et Succi) et la bibliothèque Python qlbm avant de t’y fier.

2. Les outils en circuits (rappel express)
Qubit : $\alpha|0\rangle+\beta|1\rangle$, avec $|\alpha|^2+|\beta|^2=1$. Un registre de $n$ qubits a $2^n$ amplitudes.
Hadamard $H$ : $|0\rangle\mapsto\frac{|0\rangle+|1\rangle}{\sqrt2}$, $|1\rangle\mapsto\frac{|0\rangle-|1\rangle}{\sqrt2}$. Il change de base entre “somme” et “différence”.
CNOT : inverse la cible si le contrôle vaut 1. Toffoli / mcx : même chose avec plusieurs contrôles (un grand ET logique).
$R_y(\theta)$ : $R_y(\theta)|0\rangle=\cos\frac\theta2|0\rangle+\sin\frac\theta2|1\rangle$. C’est un “potentiomètre” qui règle des amplitudes.
Ancilla + post-sélection : on ajoute un qubit auxiliaire, on applique un circuit unitaire, puis on ne garde que les mesures où l’ancilla vaut 0. L’opération effective sur les autres qubits est alors non unitaire. C’est le même procédé que dans HHL.
3. Le modèle choisi : D1Q2 (le plus simple possible)

Une dimension, deux vitesses $c_\pm=\pm1$, donc deux populations $f_+(x,t)$ et $f_-(x,t)$ sur $N$ sites avec conditions périodiques. On simule la diffusion d’une bosse de densité.

Densité et “courant” :

$$\rho = f_+ + f_-,\qquad j = f_+ - f_- .$$
3.1 Les deux étapes de la LBM

Collision (locale, sur chaque site, avec $\omega = 1/\tau$ le taux de relaxation) :

$$f_i^* = f_i-\omega\,(f_i-f_i^{eq}),\qquad f_\pm^{eq}=\frac{\rho}{2}\quad(\text{vitesse du fluide nulle}).$$

Transport (streaming) : chaque population saute d’un site selon sa vitesse :

$$f_+(x,t+1)=f_+^*(x-1,t),\qquad f_-(x,t+1)=f_-^*(x+1,t).$$
3.2 Résultat physique : une équation de diffusion

En variables $\rho, j$, la collision laisse $\rho$ inchangée et donne $j^*=(1-\omega)\,j$. Avec $f_\pm^*=\frac{\rho\pm j^*}{2}$, le transport donne

$$\rho(x,t{+}1)=\tfrac12\big[\rho(x{-}1)+\rho(x{+}1)\big]+\tfrac12\big[j^*(x{-}1)-j^*(x{+}1)\big],$$
$$j(x,t{+}1)=\tfrac12\big[\rho(x{-}1)-\rho(x{+}1)\big]+\tfrac12\big[j^*(x{-}1)+j^*(x{+}1)\big].$$

Développement de Taylor à l’ordre le plus bas :

première équation : $\partial_t\rho\approx \tfrac12\partial_x^2\rho-\partial_x j^*$ ;
seconde, en régime quasi stationnaire ($j(t{+}1)\approx j$) : $j\approx-\partial_x\rho+(1-\omega)\,j$, donc $j\approx-\partial_x\rho/\omega$ et $j^*\approx-\frac{1-\omega}{\omega}\partial_x\rho$.

On obtient

$$\boxed{\partial_t\rho = D\,\partial_x^2\rho,\qquad D=\tfrac12+\frac{1-\omega}{\omega}\cdot 1 -\ 1+\ 1\ \Longrightarrow\ D=\tau-\tfrac12.}$$

Pour éviter toute ambiguïté, le calcul est : $\tfrac12+\frac{1-\omega}{\omega}=\tfrac12+\tau-1=\tau-\tfrac12$. Vérification à $\tau=1$ : $f^*_\pm=\rho/2$, donc chaque pas est une marche aléatoire $\pm1$ et la variance croît de 1 par pas, soit $2Dt=t$ avec $D=\tfrac12$. ✓

4. Le passage au quantique
4.1 Encodage en amplitudes

On stocke les populations dans les amplitudes de $1+n$ qubits (un qubit de direction $d$, $n$ qubits de position) :

$$|\psi(t)\rangle=\frac1{\|f\|_2}\sum_{x=0}^{N-1}\Big(f_+(x,t)\,|0\rangle_d+f_-(x,t)\,|1\rangle_d\Big)|x\rangle .$$

Remarque : l’état est normalisé en $\ell_2$, alors que la masse physique $\sum\rho$ est une norme $\ell_1$. On ne récupère donc que la forme de la solution, à un facteur près.

4.2 Le transport est unitaire
$$S=|0\rangle\langle0|_d\otimes T_{+1}\;+\;|1\rangle\langle1|_d\otimes T_{-1},\qquad T_{\pm1}|x\rangle=|x\pm1 \bmod N\rangle .$$

$T_{\pm1}$ permute les états de base, donc c’est une matrice unitaire. Implémentation : un compteur contrôlé. Pour ajouter 1, le bit $i$ bascule si tous les bits plus bas valent 1 (exemple $011+1=100$). On applique donc, du bit de poids fort vers le bit de poids faible, une porte mcx (contrôles : $d$ et les bits inférieurs), puis un CNOT sur le bit 0. Le décrément est la même séquence à l’envers. La direction $d$ sert de contrôle : $d=0$ incrémente, $d=1$ décrémente.

4.3 La collision n’est pas unitaire, mais on peut la “cacher” dans une ancilla

La collision agit de la même façon sur chaque site, sur le qubit de direction seulement. Sur le vecteur $(f_+,f_-)^T$ :

$$K=(1-\omega)I+\frac{\omega}{2}\begin{pmatrix}1&1\\1&1\end{pmatrix}.$$

Le vecteur $(1,1)/\sqrt2$ (le mode $\rho$) a la valeur propre $1$, et $(1,-1)/\sqrt2$ (le mode $j$) a la valeur propre $1-\omega$. Dans la base de Hadamard,

$$K=H\,\mathrm{diag}(1,\;1-\omega)\,H .$$

Seul $\mathrm{diag}(1,1-\omega)$ est non unitaire (il “éteint” le mode $j$). On l’obtient avec une ancilla et une rotation contrôlée par le qubit de direction (après $H$) :

$$|1\rangle_d|0\rangle_a\;\mapsto\;|1\rangle_d\big(\cos\tfrac\theta2|0\rangle_a+\sin\tfrac\theta2|1\rangle_a\big),\qquad |0\rangle_d|0\rangle_a\mapsto|0\rangle_d|0\rangle_a .$$

On choisit $\cos\frac\theta2=1-\omega$, soit $\theta=2\arccos(1-\omega)$. Si on post-sélectionne $a=0$, les amplitudes du mode $|0\rangle_d$ sont inchangées et celles du mode $|1\rangle_d$ sont multipliées par $1-\omega$. C’est exactement $\mathrm{diag}(1,1-\omega)$. Ce choix demande $0\le1-\omega\le1$, c’est-à-dire $\tau\ge1$. Pour $\tau<1$, il faut en plus un signe $-$ (une porte $Z$), que je n’inclus pas.

Un pas de temps complet est donc $\;S\cdot(I_x\otimes H\,\mathrm{CRy}(\theta)\,H)$ avec une ancilla neuve par pas, et on post-sélectionne toutes les ancillas à 0.

4.4 Le coût caché

À chaque pas, l’état perd la part de son énergie portée par le mode $j$, donc la probabilité de succès $p_T=\|K^{\,T}\text{-évolué}\|^2/\|f_0\|^2$ décroît. Pour $T$ grand, il faut soit répéter beaucoup, soit amplifier l’amplitude, soit réencoder régulièrement. C’est une des raisons pour lesquelles la QLBM sur des fluides réels reste un sujet de recherche.

Cas particulier : avec $\omega=0$, on a $K=I$, donc plus d’ancilla : c’est la QLBM sans collision (transport pur), entièrement unitaire.